In [1]:
import matplotlib
matplotlib.use("Agg")
from scipy.integrate import odeint
from sympy import var, I
import numpy as np
import matplotlib.pyplot as plt
import math
import h5py

import importlib
import os
import sys
import re
import glob

In [2]:
path_openket = "//home//ultrxvioletx//GitHub//openket"
if path_openket not in sys.path:
    sys.path.append(path_openket)
# elimina el caché de func.py para que no lea los datos anteriores
if 'func' in sys.modules:
    del sys.modules['func']
if os.path.exists('func.py'):
    os.remove('func.py')

from openket.core.diracobject import Ket, Bra, Operator, AnnihilationOperator, CreationOperator
from openket.core.evolution import build_ode, sym2num, init_state
from openket.core.metrics import dag, comm, ptrace, trace, normalize, sub_qexpr, op2dict, qmatrix

import style
importlib.reload(style)
from style import *
set_style()

2026-04-27 14:38:04,484 - openket - INFO - openket v0.1.0 initialized successfully.


In [3]:
def convergencia(tiempo, fotones, tol=1e-3, porcentaje=0.1):
    npoints = int(len(tiempo) * porcentaje)
    if npoints < 2:
        return False, np.nan # no hay suficientes puntos
    fotones = fotones[-npoints:]
    tiempo = tiempo[-npoints:]
    df = np.gradient(fotones, tiempo)
    df_mean = np.mean(np.abs(df))
    return df_mean < tol, df_mean
def procesafile(path):
    global detunings, Nss, non_converged, attrs
    attrs = {}
    detunings = []
    Nss = []
    
    non_converged = []
    files = glob.glob(os.path.join(path,'*.h5'))
    for i,file in enumerate(files):
        with h5py.File(file, 'r') as f:
            dataset = os.path.basename(file).replace('.h5', '')
            if i == 0:
                attrs = dict(f[dataset].attrs)

            detuning = f[dataset].attrs[f'detuning']
            tt = f[dataset].attrs['t']
            tiempo = np.linspace(float(tt[0]), float(tt[1]), int(tt[2]))
            
            if dataset not in f:
                print(f"  no se encontró el dataset '{dataset}' en el archivo. Saltando.")
                continue

            detunings.append(detuning)
            rho = f[dataset][:]
            N_expect = rho[:, 0]

            # promediamos el último 25% de los puntos
            npoints = int(rho.shape[0] * 0.25)
            Nss.append(np.mean(N_expect[-npoints:]))
            
            is_converged, derivative = convergencia(tiempo, N_expect)
            if not is_converged:
                non_converged.append([detuning, np.mean(N_expect[-npoints:])])
    
    # ordena por detuning para mejorar la gráfica
    detunings = np.array(detunings)
    sorti = np.argsort(detunings)
    
    detunings = detunings[sorti]
    Nss = np.array(Nss)[sorti]

## deltas cavidad

In [5]:
path = os.path.join("detunings","cavidad")
procesafile(path)

cut=3
mask = (detunings >= -cut) & (detunings <= cut)
plt.plot(detunings[mask], Nss[mask], 's-', markersize=5, linewidth=1, color=colores["fotones"])
if non_converged:
    dtn = [ele[0] for ele in non_converged]
    nmean = [ele[1] for ele in non_converged]
    #plt.scatter(dtn, nmean, color="red")
plt.axvline(0, color="gray", linestyle="--", linewidth=1, alpha=0.5)
plt.xlabel(r"$\Delta / \kappa$")
plt.ylabel(r"Número medio de fotones $\langle N \rangle_{ss}$")
ax = plt.gca()
xval=3.5
format_ax(ax, xstep=1, ystep=2, ymax=max(Nss)+1, xlim=(-xval,xval))
plt.savefig(f"../figuras/cavidad_fotones.png")
plt.close()

for file in os.listdir("."):
    if file.endswith(".py") and file != "style.py":
        os.remove(file)

In [5]:
attrs

{'detuning': np.float64(3.6),
 'id': np.int32(77),
 'kappa': np.float64(1.0),
 'n': np.int32(10),
 'rabi_p': np.float64(1.0),
 't': array(['0', '10.0', '1000'], dtype=object)}

## tiempo cavidad

In [4]:
filename = "cavidad"

hbar = 1.0
rabi_b = 1.0 # tasa del bombeo, proporcional a la amplitud del campo eléctrico del láser
detuning = 0.0 # detuning entre frecuencia de la cavidad y frecuencia del bombeo

kappa = 1.0
nmax = 15 # truncación del espacio de Hilbert (número de estados de Fock)
dt = 1000
t = np.linspace(0, 15, dt)

base = [Ket(i, "cavidad") for i in range(nmax)]
rho = Operator("R")
a = AnnihilationOperator("cavidad", nmax-1)
aa = CreationOperator("cavidad", nmax-1)

H_c = hbar * detuning * (aa*a + 1/2) # Hamiltoniano del oscilador armónico cuántico (cavidad)
H_b = I*hbar * rabi_b * (aa - a) # Hamiltoniano del bombeo (láser)
H = H_c + H_b # Hamiltoniano total
# ecuación de movimiento de Lindblad
rdot = -I/hbar * comm(H,rho) + (kappa/2)*(2*a*rho*aa - aa*a*rho - rho*aa*a)

build_ode(
    rho=rho,
    rdot=rdot,
    basis=base,
    filetype="Scipy",
    filename=f"func{filename}.py",
    dictname=f"dic{filename}.py"
)

# leer módulo dic.py
funcname=f"func{filename}"
if funcname in sys.modules:
    del sys.modules[funcname]
modulo = importlib.import_module(funcname)
modulo = importlib.reload(modulo)
f = modulo.f
dic = modulo.dic
# convertir condiciones iniciales simbólicas -> numéricas
rho0 = Ket(0,"cavidad") * dag(Ket(0,"cavidad"))
init_conditions = init_state(rho=rho, rho0=rho0, basis=base, dic=dic)
# solución numérica
rho_solution = odeint(f, init_conditions, t)

# Valores esperados numéricos
# definición simbólica de los observables
N = aa * a # operador de número
N_symb = sub_qexpr(qexpr=trace(rho * N, basis=base), dic=dic)
N_expect = sym2num(sol=rho_solution, symbexpr=N_symb)
N_expect = np.real(N_expect)

# Valores esperados analíticos
alpha_ss = rabi_b / (kappa/2 + 1j*detuning) #amplitud compleja del estado estacionario (cuando t->oo)
alpha = alpha_ss*(1 - np.exp(-(kappa/2 + 1j*detuning) * t)) #amplitud compleja del campo coherente en la cavidad
N_analytical = np.abs(alpha)**2

In [9]:
step=20
plt.scatter(t[::step], N_expect[::step], label=r'numérico', color=colores["fotones"], s=15, marker='s', zorder=2)
plt.plot(t, N_analytical, label=r'analítico', color=colores["fotones"], linewidth=1, zorder=1)
plt.xlabel(r"tiempo $\kappa^{-1}$")
plt.ylabel(rf"Número medio de fotones ${latex["Nt"]}$")
plt.axhline(4, color="gray", linestyle="--", linewidth=0.5)
ax = plt.gca()
format_ax(ax, xstep=2, ystep=2, ymax=max(N_expect)+1)
plt.legend()
plt.savefig(f"../figuras/cavidad.png")
plt.close()

for file in os.listdir("."):
    if file.endswith(".py") and file != "style.py":
        os.remove(file)